# ETL RAW -> Silver

Projeto: **Mapa da Cidadania e Acesso à Informação no DF**.

Este notebook segue a organização do `etl_raw_to_silver_Exemplo.ipynb`: configuração, funções auxiliares, extração, transformação, validação e carga da camada Silver.

A saída final é uma BigTable em granularidade **Região Administrativa + ano**. Como a base Participa DF/LAI não possui campo territorial confiável de Região Administrativa, seus indicadores são agregados no nível **Distrito Federal + ano** e repetidos para cada RA do mesmo ano, sem inferência territorial artificial.


## 1. Configuração e Importações

Define bibliotecas, caminhos da arquitetura medalhão e parâmetros globais do projeto.


In [1]:
from pathlib import Path
import re
import unicodedata
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Transformer":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "Data Layer" / "raw"
SILVER_DIR = PROJECT_ROOT / "Data Layer" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)
ANOS_PROJETO = [2023, 2024, 2025]
CODIGOS_ESPECIAIS_PDAD = {77777, 88888, 99999, 77777.0, 88888.0, 99999.0, "77777", "88888", "99999"}

print('Configuração carregada')
print(f'Projeto: {PROJECT_ROOT}')
print(f'Entrada RAW: {RAW_DIR}')
print(f'Saída Silver: {SILVER_DIR}')
print(f'Anos do projeto: {ANOS_PROJETO}')


Configuração carregada
Projeto: /home/iggor/Documentos/Mapa-da-Cidadania-e-Acesso-a-Informacao-DF
Entrada RAW: /home/iggor/Documentos/Mapa-da-Cidadania-e-Acesso-a-Informacao-DF/Data Layer/raw
Saída Silver: /home/iggor/Documentos/Mapa-da-Cidadania-e-Acesso-a-Informacao-DF/Data Layer/silver
Anos do projeto: [2023, 2024, 2025]


## 2. Funções Auxiliares

Funções reutilizáveis para padronização de colunas, normalização textual, leitura resiliente de CSVs, conversão de tipos, tratamento de códigos especiais da PDAD e cálculos ponderados.


In [2]:
def remover_acentos(texto):
    if pd.isna(texto): return texto
    return "".join(c for c in unicodedata.normalize("NFKD", str(texto)) if not unicodedata.combining(c))

def padronizar_colunas(df):
    cols=[]; usados={}
    for col in df.columns:
        novo=remover_acentos(col).lower().strip().replace('.', '_')
        novo=re.sub(r"\s+", "_", novo)
        novo=re.sub(r"[^a-z0-9_]+", "_", novo)
        novo=re.sub(r"_+", "_", novo).strip('_') or 'coluna'
        base=novo
        if base in usados:
            usados[base]+=1; novo=f"{base}_{usados[base]}"
        else:
            usados[base]=0
        cols.append(novo)
    out=df.copy(); out.columns=cols; return out

def normalizar_chave(texto):
    if pd.isna(texto): return np.nan
    texto=remover_acentos(texto).lower().strip()
    texto=re.sub(r"[^a-z0-9]+", " ", texto)
    texto=re.sub(r"\s+", " ", texto).strip()
    aliases_ra = {
        'sudoeste octogonal': 'sudoeste e octogonal',
        'sol nascente por do sol': 'sol nascente por do sol',
    }
    return aliases_ra.get(texto, texto)

def normalizar_nome_orgao(texto):
    if pd.isna(texto) or str(texto).strip()=="": return "Não informado"
    return re.sub(r"\s+", " ", str(texto)).strip()

def ler_csv_auto(caminho, nrows=None):
    melhor=None; erros=[]
    for kwargs in [{"sep":";","encoding":"utf-8-sig"},{"sep":";","encoding":"latin1"},{"sep":",","encoding":"utf-8-sig"},{"sep":",","encoding":"latin1"}]:
        try:
            df=pd.read_csv(caminho, nrows=nrows, **kwargs)
            if melhor is None or len(df.columns)>len(melhor.columns): melhor=df
            if len(df.columns)>1: return df
        except Exception as exc:
            erros.append(f"{kwargs}: {exc}")
    if melhor is not None: return melhor
    raise RuntimeError("Não foi possível ler CSV: "+" | ".join(erros))

def converter_colunas_data(df):
    out=df.copy()
    for col in [c for c in out.columns if c.startswith('dt_') or '_dt_' in c or 'data' in c]:
        out[col]=pd.to_datetime(out[col], dayfirst=True, errors='coerce')
    return out

def para_numero(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors='coerce')
    s=serie.astype(str).str.strip().replace({'nan':np.nan,'None':np.nan,'':np.nan})
    tem_virgula=s.str.contains(',', na=False)
    s=s.mask(tem_virgula, s[tem_virgula].str.replace('.', '', regex=False).str.replace(',', '.', regex=False))
    return pd.to_numeric(s, errors='coerce')

def tratar_codigos_especiais(df, preservar=None):
    preservar=set(preservar or []); out=df.copy()
    for col in out.columns:
        if col in preservar: continue
        if pd.api.types.is_numeric_dtype(out[col]): out[col]=out[col].replace(list(CODIGOS_ESPECIAIS_PDAD), np.nan)
        else: out[col]=out[col].replace(['77777','88888','99999'], np.nan)
    return out

def media_ponderada(valor, peso):
    dados=pd.DataFrame({'valor':valor,'peso':peso}).dropna(); dados=dados[dados['peso']>0]
    return np.nan if dados.empty or dados['peso'].sum()==0 else np.average(dados['valor'], weights=dados['peso'])

def percentual_ponderado(condicao, peso):
    dados=pd.DataFrame({'condicao':condicao,'peso':peso}).dropna(); dados=dados[dados['peso']>0]
    return np.nan if dados.empty or dados['peso'].sum()==0 else 100*dados.loc[dados['condicao'].astype(bool),'peso'].sum()/dados['peso'].sum()


## 3. Carregando os Dados Brutos (EXTRACT)

Leitura das fontes RAW usadas na Silver: dicionário PDAD, LAI/Participa DF, moradores, domicílios e projeções populacionais.


In [3]:
# Dicionário PDAD
anexo_localidade=padronizar_colunas(pd.read_excel(RAW_DIR/'dicionario_de_variaveis_pdada_2024_público.xlsx', sheet_name='anexo_1')).ffill()
anexo_localidade['valor']=pd.to_numeric(anexo_localidade['valor'], errors='coerce')
mapa_localidade=dict(zip(anexo_localidade['valor'], anexo_localidade['descricao_do_valor']))


## 4. Transformação dos Dados (TRANSFORM)

Padronização e enriquecimento das bases intermediárias antes da consolidação da BigTable Silver.


### 4.1 LAI/Participa DF - Pedidos

Padroniza datas, ano/mês, unidade demandada, tempo de resposta e situação de resposta.


In [4]:
# LAI pedidos
pedidos=converter_colunas_data(padronizar_colunas(ler_csv_auto(RAW_DIR/'Participa DF _LAI_ - Pedidos.csv')))
pedidos['ano']=pd.to_numeric(pedidos['nr_ano_pedido'], errors='coerce').astype('Int64') if 'nr_ano_pedido' in pedidos.columns else pedidos.get('dt_pedido', pd.Series(pd.NaT,index=pedidos.index)).dt.year.astype('Int64')
col_data_pedido='dt_pedido' if 'dt_pedido' in pedidos.columns else 'dt_data_pedido'
if col_data_pedido in pedidos.columns:
    pedidos['mes']=pedidos[col_data_pedido].dt.month.astype('Int64'); pedidos['ano_mes']=pedidos[col_data_pedido].dt.to_period('M').astype(str)
else:
    pedidos['mes']=pd.NA; pedidos['ano_mes']=pedidos['ano'].astype(str)
col_resposta='respostas_dt_resposta'
pedidos['dias_resposta']=((pedidos[col_resposta]-pedidos[col_data_pedido]).dt.total_seconds()/86400) if col_data_pedido in pedidos.columns and col_resposta in pedidos.columns else np.nan
pedidos['pedido_sem_resposta']=pedidos.get(col_resposta, pd.Series(pd.NaT,index=pedidos.index)).isna()
if 'ds_unidade' in pedidos.columns: pedidos['ds_unidade']=pedidos['ds_unidade'].map(normalizar_nome_orgao)
for col in [c for c in pedidos.columns if 'classificacao' in c or 'tipo_resposta' in c or 'situacao' in c]: pedidos[col]=pedidos[col].fillna('Não informado')
pedidos=pedidos[pedidos['ano'].isin(ANOS_PROJETO)].copy()
# DataFrame intermediário: LAI pedidos não é salvo como CSV final da Silver.


### 4.2 LAI/Participa DF - Recursos

Padroniza datas, ano/mês, classificações e remove duplicidades.


In [5]:
# LAI recursos
recursos=converter_colunas_data(padronizar_colunas(ler_csv_auto(RAW_DIR/'Participa DF _LAI_ - Recursos.csv')))
if 'dt_recurso' in recursos.columns:
    recursos['ano']=recursos['dt_recurso'].dt.year.astype('Int64'); recursos['mes']=recursos['dt_recurso'].dt.month.astype('Int64'); recursos['ano_mes']=recursos['dt_recurso'].dt.to_period('M').astype(str)
else:
    recursos['ano']=pd.NA; recursos['mes']=pd.NA; recursos['ano_mes']=pd.NA
recursos=recursos[recursos['ano'].isin(ANOS_PROJETO)].drop_duplicates().copy()
for col in [c for c in recursos.columns if 'classificacao' in c or 'tipo_resposta' in c or 'instancia' in c]: recursos[col]=recursos[col].fillna('Não informado')
# DataFrame intermediário: LAI recursos não é salvo como CSV final da Silver.


### 4.3 LAI/Participa DF - Pesquisa de Satisfação

Padroniza datas e categorias textuais da pesquisa de satisfação.


In [6]:
# LAI satisfação
satisfacao=converter_colunas_data(padronizar_colunas(ler_csv_auto(RAW_DIR/'Participa DF _LAI_ - Pesquisa de Satisfação.csv')))
col_data_satisfacao=next((c for c in satisfacao.columns if c.startswith('dt_') or 'data' in c), None)
if col_data_satisfacao:
    satisfacao['ano']=satisfacao[col_data_satisfacao].dt.year.astype('Int64'); satisfacao['mes']=satisfacao[col_data_satisfacao].dt.month.astype('Int64'); satisfacao['ano_mes']=satisfacao[col_data_satisfacao].dt.to_period('M').astype(str)
    satisfacao=satisfacao[satisfacao['ano'].isin(ANOS_PROJETO)].copy()
else:
    satisfacao['ano']=pd.NA; satisfacao['mes']=pd.NA; satisfacao['ano_mes']=pd.NA
for col in satisfacao.select_dtypes(include='object').columns: satisfacao[col]=satisfacao[col].fillna('Não informado')
# DataFrame intermediário: LAI satisfação não é salvo como CSV final da Silver.


/tmp/ipykernel_81858/1234845639.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in satisfacao.select_dtypes(include='object').columns: satisfacao[col]=satisfacao[col].fillna('Não informado')


### 4.4 PDAD - Moradores

Converte variáveis numéricas, trata códigos especiais e cria a chave padronizada de Região Administrativa.


In [7]:
# PDAD moradores
moradores=padronizar_colunas(ler_csv_auto(RAW_DIR/'moradores.csv'))
if 'peso_mor' in moradores.columns: moradores['peso_mor']=para_numero(moradores['peso_mor'])
for col in ['idade_calculada','renda_ind','renda_ind_r','escolaridade','localidade']:
    if col in moradores.columns: moradores[col]=para_numero(moradores[col])
moradores=tratar_codigos_especiais(moradores, preservar={'localidade','setor_distrito','a01uf','a01nficha','morador_id','index'})
if 'localidade' in moradores.columns:
    moradores['regiao_administrativa']=moradores['localidade'].map(mapa_localidade).fillna(moradores['localidade'].astype(str))
    moradores['ra_chave']=moradores['regiao_administrativa'].map(normalizar_chave)
# DataFrame intermediário: PDAD moradores não é salvo como CSV final da Silver.


/tmp/ipykernel_81858/1637743223.py:39: DtypeWarning: Columns (0: E03_1_1, 1: E03_2_1) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(caminho, nrows=nrows, **kwargs)


/tmp/ipykernel_81858/2787312226.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  moradores['regiao_administrativa']=moradores['localidade'].map(mapa_localidade).fillna(moradores['localidade'].astype(str))
/tmp/ipykernel_81858/2787312226.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  moradores['ra_chave']=moradores['regiao_administrativa'].map(normalizar_chave)


### 4.5 PDAD - Domicílios

Converte variáveis numéricas, trata códigos especiais e cria a chave padronizada de Região Administrativa.


In [8]:
# PDAD domicílios
domicilios=padronizar_colunas(pd.read_excel(RAW_DIR/'domicilios.xlsx'))
if 'peso_dom' in domicilios.columns: domicilios['peso_dom']=para_numero(domicilios['peso_dom'])
for col in domicilios.columns:
    if col!='starttime' and domicilios[col].dtype=='object':
        convertido=para_numero(domicilios[col])
        if convertido.notna().mean()>0.8: domicilios[col]=convertido
for col in ['localidade','a01npessoas','b03','c05','renda_domiciliar','renda_domiciliar_r','renda_domiciliar_pc','renda_domiciliar_pc_r']:
    if col in domicilios.columns: domicilios[col]=para_numero(domicilios[col])
domicilios=tratar_codigos_especiais(domicilios, preservar={'localidade','setor_distrito','a01uf','a01nficha'})
if 'localidade' in domicilios.columns:
    domicilios['regiao_administrativa']=domicilios['localidade'].map(mapa_localidade).fillna(domicilios['localidade'].astype(str))
    domicilios['ra_chave']=domicilios['regiao_administrativa'].map(normalizar_chave)
# DataFrame intermediário: PDAD domicílios não é salvo como CSV final da Silver.


/tmp/ipykernel_81858/2839186004.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  domicilios['regiao_administrativa']=domicilios['localidade'].map(mapa_localidade).fillna(domicilios['localidade'].astype(str))
/tmp/ipykernel_81858/2839186004.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  domicilios['ra_chave']=domicilios['regiao_administrativa'].map(normalizar_chave)


### 4.6 Projeções Populacionais

Transforma as planilhas por Região Administrativa em uma base longa e consolida população por sexo e faixa etária.


In [9]:
# Projeções populacionais
def classificar_faixa_etaria(valor):
    texto=str(valor).strip().lower()
    if texto in {'nan', 'total'}:
        return np.nan
    if '+' in texto:
        inicio=pd.to_numeric(re.findall(r'\d+', texto)[0], errors='coerce')
    else:
        nums=re.findall(r'\d+', texto)
        inicio=pd.to_numeric(nums[0], errors='coerce') if nums else np.nan
    if pd.isna(inicio):
        return np.nan
    if inicio <= 14:
        return '0-14'
    if inicio <= 59:
        return '15-59'
    return '60+'

def extrair_populacao_sheet(caminho, sheet_name):
    bruto=pd.read_excel(caminho, sheet_name=sheet_name, header=None)
    anos=bruto.iloc[3].ffill(); sexos=bruto.iloc[4]
    dados=bruto.iloc[5:].copy().rename(columns={0:'idade'})
    dados['faixa_etaria']=dados['idade'].map(classificar_faixa_etaria)
    dados=dados[dados['faixa_etaria'].notna()].copy()
    partes=[]
    for col in dados.columns[1:]:
        if col == 'faixa_etaria':
            continue
        ano=pd.to_numeric(anos[col], errors='coerce'); sexo=sexos[col]
        if pd.isna(ano) or pd.isna(sexo): continue
        sexo=str(sexo).strip()
        if sexo not in {'Total','Homens','Mulheres'}: continue
        tmp=dados[['faixa_etaria',col]].rename(columns={col:'populacao'})
        tmp['ano']=int(ano); tmp['sexo']=sexo; partes.append(tmp)
    if not partes: return pd.DataFrame(columns=['regiao_administrativa','ano','sexo','faixa_etaria','populacao'])
    longo=pd.concat(partes, ignore_index=True); longo['populacao']=pd.to_numeric(longo['populacao'], errors='coerce')
    longo['regiao_administrativa']=sheet_name
    return longo[['regiao_administrativa','ano','sexo','faixa_etaria','populacao']]

pop_path=RAW_DIR/'Dados - Projeções populacionais por Região Administrativa, sexo e faixa etária 2020-2030_1.xlsx'
xl_pop=pd.ExcelFile(pop_path); sheets_ra=[s for s in xl_pop.sheet_names if s not in {'DF', "RA's"}]
pop_longa=pd.concat([extrair_populacao_sheet(pop_path, s) for s in sheets_ra], ignore_index=True)
pop_longa=pop_longa[pop_longa['ano'].isin(ANOS_PROJETO)].copy(); pop_longa['ra_chave']=pop_longa['regiao_administrativa'].map(normalizar_chave)
pop_total=pop_longa[pop_longa['sexo']=='Total'].pivot_table(index=['regiao_administrativa','ra_chave','ano'], columns='faixa_etaria', values='populacao', aggfunc='sum').reset_index().rename(columns={'0-14':'populacao_0_14','15-59':'populacao_15_59','60+':'populacao_60_mais','60':'populacao_60_mais'})
for col in ['populacao_0_14','populacao_15_59','populacao_60_mais']:
    if col not in pop_total.columns:
        pop_total[col]=0
pop_sexo=pop_longa[pop_longa['sexo'].isin(['Homens','Mulheres'])].groupby(['regiao_administrativa','ra_chave','ano','sexo'], as_index=False)['populacao'].sum().pivot_table(index=['regiao_administrativa','ra_chave','ano'], columns='sexo', values='populacao', aggfunc='sum').reset_index().rename(columns={'Homens':'populacao_masculina','Mulheres':'populacao_feminina'})
for col in ['populacao_masculina','populacao_feminina']:
    if col not in pop_sexo.columns:
        pop_sexo[col]=np.nan
pop_ra_ano=pop_total.merge(pop_sexo, on=['regiao_administrativa','ra_chave','ano'], how='left')
pop_ra_ano['populacao_total']=pop_ra_ano[['populacao_0_14','populacao_15_59','populacao_60_mais']].sum(axis=1)
pop_ra_ano=pop_ra_ano[['ano','regiao_administrativa','ra_chave','populacao_total','populacao_masculina','populacao_feminina','populacao_0_14','populacao_15_59','populacao_60_mais']].sort_values(['regiao_administrativa','ano']).reset_index(drop=True)
# DataFrame intermediário: população RA/ano não é salva como CSV final da Silver.


## 5. Agregações Analíticas

Criação dos indicadores ponderados da PDAD e dos indicadores anuais da LAI no nível Distrito Federal.


### 5.1 Indicadores PDAD por Região Administrativa

Calcula renda, idade, escolaridade, moradores por domicílio, internet e condição de ocupação usando pesos amostrais.


In [10]:
# Indicadores PDAD ponderados
indic_mor=[]
for chave, grupo in moradores.groupby('ra_chave', dropna=True):
    indic_mor.append({'ra_chave':chave,'renda_media_ponderada':media_ponderada(grupo.get('renda_ind_r', grupo.get('renda_ind')), grupo['peso_mor']),'idade_media_ponderada':media_ponderada(grupo.get('idade_calculada'), grupo['peso_mor']),'percentual_baixa_escolaridade':percentual_ponderado(grupo.get('escolaridade').isin([1,2,3]), grupo['peso_mor'])})
indic_mor=pd.DataFrame(indic_mor)
indic_dom=[]
for chave, grupo in domicilios.groupby('ra_chave', dropna=True):
    indic_dom.append({'ra_chave':chave,'media_moradores_por_domicilio':media_ponderada(grupo.get('a01npessoas'), grupo['peso_dom']),'percentual_domicilios_com_internet':percentual_ponderado(grupo.get('c05').isin([1,2]), grupo['peso_dom']),'percentual_domicilios_proprios':percentual_ponderado(grupo.get('b03').isin([1,2]), grupo['peso_dom']),'percentual_domicilios_alugados':percentual_ponderado(grupo.get('b03').eq(3), grupo['peso_dom'])})
indic_dom=pd.DataFrame(indic_dom)
indic_pdad=indic_mor.merge(indic_dom, on='ra_chave', how='outer')


### 5.2 Indicadores LAI por Ano no Distrito Federal

Agrega pedidos, recursos, satisfação, tempo médio de resposta, tipos de acesso e órgãos demandados por ano.


In [11]:
# Indicadores LAI no nível Distrito Federal + ano
class_col='respostas_ds_tipo_classificacao_resposta' if 'respostas_ds_tipo_classificacao_resposta' in pedidos.columns else None
org_col='ds_unidade' if 'ds_unidade' in pedidos.columns else None

def pct_classificacao_lai(serie, termo):
    if serie is None:
        return np.nan
    return 100 * serie.fillna('Não informado').str.contains(termo, case=False, na=False).mean()

agg_pedidos = {
    'lai_df_qtd_pedidos_ano': ('nr_pedido_participa', 'nunique'),
    'lai_df_tempo_medio_resposta_dias': ('dias_resposta', 'mean'),
    'lai_df_percentual_sem_resposta': ('pedido_sem_resposta', lambda s: 100 * s.mean()),
}
if org_col:
    agg_pedidos['lai_df_qtd_orgaos_demandados'] = (org_col, 'nunique')
else:
    pedidos['orgao_nao_informado'] = 'Não informado'
    org_col = 'orgao_nao_informado'
    agg_pedidos['lai_df_qtd_orgaos_demandados'] = (org_col, 'nunique')

indicadores_lai_df_ano = pedidos.groupby('ano', as_index=False).agg(**agg_pedidos)

if class_col:
    class_lai = pedidos.groupby('ano', as_index=False)[class_col].agg(
        lai_df_percentual_acesso_concedido=lambda s: pct_classificacao_lai(s, r'Acesso Concedido$|^Acesso Concedido'),
        lai_df_percentual_acesso_negado=lambda s: pct_classificacao_lai(s, r'Acesso Negado'),
        lai_df_percentual_acesso_parcial=lambda s: pct_classificacao_lai(s, r'Parcial'),
    )
else:
    class_lai = pd.DataFrame({'ano': ANOS_PROJETO})
    class_lai['lai_df_percentual_acesso_concedido'] = np.nan
    class_lai['lai_df_percentual_acesso_negado'] = np.nan
    class_lai['lai_df_percentual_acesso_parcial'] = np.nan

indicadores_lai_df_ano = indicadores_lai_df_ano.merge(class_lai, on='ano', how='left')

recursos_df_ano = recursos.groupby('ano', as_index=False).agg(
    lai_df_qtd_recursos_ano=('nr_pedido_participa', 'count'),
    lai_df_qtd_pedidos_com_recurso_ano=('nr_pedido_participa', 'nunique'),
)
indicadores_lai_df_ano = indicadores_lai_df_ano.merge(recursos_df_ano, on='ano', how='left')
indicadores_lai_df_ano['lai_df_qtd_recursos_ano'] = indicadores_lai_df_ano['lai_df_qtd_recursos_ano'].fillna(0)
indicadores_lai_df_ano['lai_df_qtd_pedidos_com_recurso_ano'] = indicadores_lai_df_ano['lai_df_qtd_pedidos_com_recurso_ano'].fillna(0)

satisfacao_df_ano = satisfacao.dropna(subset=['ano']).groupby('ano', as_index=False).agg(
    lai_df_qtd_satisfacoes_ano=('nr_pedido_participa', 'count')
)
indicadores_lai_df_ano = indicadores_lai_df_ano.merge(satisfacao_df_ano, on='ano', how='left')
indicadores_lai_df_ano['lai_df_qtd_satisfacoes_ano'] = indicadores_lai_df_ano['lai_df_qtd_satisfacoes_ano'].fillna(0)

indicadores_lai_df_ano['lai_df_percentual_pedidos_com_recurso'] = np.where(
    indicadores_lai_df_ano['lai_df_qtd_pedidos_ano'] > 0,
    100 * indicadores_lai_df_ano['lai_df_qtd_pedidos_com_recurso_ano'] / indicadores_lai_df_ano['lai_df_qtd_pedidos_ano'],
    np.nan,
)

cols_lai_df = [
    'lai_df_qtd_pedidos_ano',
    'lai_df_qtd_recursos_ano',
    'lai_df_qtd_satisfacoes_ano',
    'lai_df_percentual_pedidos_com_recurso',
    'lai_df_tempo_medio_resposta_dias',
    'lai_df_percentual_acesso_concedido',
    'lai_df_percentual_acesso_negado',
    'lai_df_percentual_acesso_parcial',
    'lai_df_percentual_sem_resposta',
    'lai_df_qtd_orgaos_demandados',
]
indicadores_lai_df_ano = indicadores_lai_df_ano[['ano'] + cols_lai_df].sort_values('ano').reset_index(drop=True)
# DataFrame intermediário: indicadores LAI DF/ano não são salvos como CSV final da Silver.


## 6. Consolidação da BigTable Silver

Integra população, indicadores socioeconômicos e indicadores LAI DF/ano em uma tabela final RA + ano.


In [12]:
# Consolidada RA/ano com indicadores LAI DF/ano, sem LAI territorial artificial
termos_territorio={'regiao','regiao_administrativa','localidade','cidade','ra','territorio','endereco'}
colunas_territoriais_lai=[]
for col in pedidos.columns:
    tokens=set(re.split(r'_+', col))
    if col in termos_territorio or tokens.intersection(termos_territorio):
        colunas_territoriais_lai.append(col)
print('Colunas territoriais LAI encontradas:', colunas_territoriais_lai)
mapa_cidadania=pop_ra_ano.merge(indic_pdad, on='ra_chave', how='left').drop(columns=['ra_chave'])
mapa_cidadania.insert(2, 'ano_referencia_pdad', 2024)
mapa_cidadania=mapa_cidadania.merge(indicadores_lai_df_ano, on='ano', how='left')
ordem=['ano','regiao_administrativa','ano_referencia_pdad','populacao_total','populacao_masculina','populacao_feminina','populacao_0_14','populacao_15_59','populacao_60_mais','renda_media_ponderada','idade_media_ponderada','media_moradores_por_domicilio','percentual_baixa_escolaridade','percentual_domicilios_com_internet','percentual_domicilios_proprios','percentual_domicilios_alugados'] + cols_lai_df
mapa_cidadania=mapa_cidadania[[c for c in ordem if c in mapa_cidadania.columns]].sort_values(['regiao_administrativa','ano']).reset_index(drop=True)


Colunas territoriais LAI encontradas: []


## 7. Salvando a Camada Silver (LOAD)

Grava a BigTable final em CSV na pasta `Data Layer/silver`.


In [13]:
mapa_cidadania.to_csv(
    SILVER_DIR / 'tb_mapa_cidadania_ra_ano_silver.csv',
    index=False,
    encoding='utf-8-sig'
)
print(f"Salvo: {(SILVER_DIR / 'tb_mapa_cidadania_ra_ano_silver.csv').relative_to(PROJECT_ROOT)} | linhas={len(mapa_cidadania):,} colunas={len(mapa_cidadania.columns):,}")


Salvo: Data Layer/silver/tb_mapa_cidadania_ra_ano_silver.csv | linhas=99 colunas=26


## 8. Validação dos Dados

Valida granularidade, período, quantidade de Regiões Administrativas, duplicidades, nulos críticos e consistência metodológica dos indicadores LAI.


In [14]:
# Validações da BigTable Silver
arquivo_bigtable = SILVER_DIR / 'tb_mapa_cidadania_ra_ano_silver.csv'
assert arquivo_bigtable.exists(), 'BigTable Silver não gerada.'
assert len(mapa_cidadania) == 99, f'Quantidade inesperada de linhas: {len(mapa_cidadania)}'
assert len(mapa_cidadania.columns) == 26, f'Quantidade inesperada de colunas: {len(mapa_cidadania.columns)}'
assert mapa_cidadania['regiao_administrativa'].nunique() == 33, 'Quantidade inesperada de Regiões Administrativas.'
assert set(mapa_cidadania['ano'].dropna().astype(int).unique()) == set(ANOS_PROJETO), 'Anos fora do período esperado.'
assert set(mapa_cidadania['ano_referencia_pdad'].dropna().astype(int).unique()) == {2024}, 'Ano de referência PDAD inesperado.'
assert not mapa_cidadania.duplicated(['regiao_administrativa','ano']).any(), 'Duplicidade na granularidade RA/ano'
cols_pdad = [
    'ano_referencia_pdad',
    'renda_media_ponderada',
    'idade_media_ponderada',
    'media_moradores_por_domicilio',
    'percentual_baixa_escolaridade',
    'percentual_domicilios_com_internet',
    'percentual_domicilios_proprios',
    'percentual_domicilios_alugados',
]
assert not mapa_cidadania[cols_pdad].isna().any().any(), 'Indicadores socioeconômicos principais com nulos.'
assert not mapa_cidadania[cols_lai_df].isna().any().any(), 'Indicadores LAI DF/ano com nulos.'
variacoes_lai_por_ano = mapa_cidadania.groupby('ano')[cols_lai_df].nunique(dropna=False)
assert (variacoes_lai_por_ano == 1).all().all(), 'Indicadores LAI variam entre RAs dentro do mesmo ano.'

resumo = pd.DataFrame({
    'metrica': [
        'linhas',
        'colunas',
        'regioes_administrativas',
        'anos',
        'ano_referencia_pdad',
        'duplicidades_ra_ano',
        'linhas_com_nulos_socioeconomicos',
        'linhas_com_nulos_lai_df',
        'max_variacoes_lai_df_por_ano',
    ],
    'valor': [
        len(mapa_cidadania),
        len(mapa_cidadania.columns),
        mapa_cidadania['regiao_administrativa'].nunique(),
        ', '.join(map(str, sorted(mapa_cidadania['ano'].dropna().astype(int).unique()))),
        ', '.join(map(str, sorted(mapa_cidadania['ano_referencia_pdad'].dropna().astype(int).unique()))),
        int(mapa_cidadania.duplicated(['regiao_administrativa','ano']).sum()),
        int(mapa_cidadania[cols_pdad].isna().any(axis=1).sum()),
        int(mapa_cidadania[cols_lai_df].isna().any(axis=1).sum()),
        int(variacoes_lai_por_ano.max().max()),
    ]
})
try:
    display(resumo)
except NameError:
    print(resumo.to_string(index=False))


,metrica,valor
0,linhas,99
1,colunas,26
2,regioes_administrativas,33
3,anos,"2023, 2024, 2025"
4,ano_referencia_pdad,2024
5,duplicidades_ra_ano,0
6,linhas_com_nulos_socioeconomicos,0
7,linhas_com_nulos_lai_df,0
8,max_variacoes_lai_df_por_ano,1
